In [1]:
import pyart
import numpy as np
from scipy.signal import filtfilt, butter


## You are using the Python ARM Radar Toolkit (Py-ART), an open source
## library for working with weather radar data. Py-ART is partly
## supported by the U.S. Department of Energy as part of the Atmospheric
## Radiation Measurement (ARM) Climate Research Facility, an Office of
## Science user facility.
##
## If you use this software to prepare a publication, please cite:
##
##     JJ Helmus and SM Collis, JORS 2016, doi: 10.5334/jors.119



In [2]:
def lowpass_filter(data, cutoff, fs, order=4):

    """
    Helper function that overlays a lowpass filter on the radial velocity field
    """
    
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    b, a = butter(order, normal_cutoff, btype='low', analog=False)
    
    return filtfilt(b, a, data)

In [3]:
# More helper functions

# Function that finds the Volume Coverage Pattern contained within the radar file  
def get_vcp(radar):
    vcp = radar.fixed_angle["data"]
    return np.asarray(vcp, dtype=radar.elevation['data'].dtype)

# Function that iterates through the elevations angle and masks the repeated angles 
def unique_sweeps_by_elevation_angle(radar, tolerance=0.05):
    vcp = get_vcp(radar)
    n_sweeps = len(vcp)

    # mask to include all sweeps initially 
    keep_mask = np.ones(n_sweeps, dtype=bool)

    # Remove repeated low-level sweeps (like 0.3°)
    seen = set()
    for i, angle in enumerate(vcp):
        # Find close matches already seen
        if any(np.isclose(angle, s, atol=tolerance) for s in seen):
            keep_mask[i] = False
        else:
            seen.add(angle)

    # Remove the last two sweeps (highest elevations)
    keep_mask[-2:] = False

    # Return the indices of sweeps to keep
    return np.where(keep_mask)[0]

In [4]:
# Computing residual velocity for an example time

rad_file = './SampleData/cfrad.20240913_003234.186_to_20240913_003916.184_KNQA_SUR.nc' # QCd using Solo3

radar = pyart.io.read(rad_file)

# Removing repeated elevation angles in radar file
unq_swp = unique_sweeps_by_elevation_angle(radar)

# Radar data with only unique sweeps
radar_sub = radar.extract_sweeps(unq_swp)

ranges = radar_sub.range['data']     # Replace "Edc_VEL3" with whatever your quality controlled radial velocity variable is
radial_velocity_sm = np.zeros((radar_sub.fields['Edc_VEL3']['data'].shape[0], 1832)) # Creating empty array for low-pass ("mean") radial velocity 
pertubations = np.zeros((radar_sub.fields['Edc_VEL3']['data'].shape[0], 1832)) # Creating an empty array for residual velocity 

# Residual velocity = radial_velocity - radial_velocity_sm (Radial velocity - mean)


for sweep_idx in range(radar_sub.nsweeps): # Loop through every sweep
    for k, target_range in enumerate(ranges): # Loop through every range 
        # ---> Radial velocity at a particular range and sweep is a sinusoidal function of azimuth

        # Finding where the sweep starts and ends in the data
        start_idx = radar_sub.sweep_start_ray_index['data'][sweep_idx]
        end_idx = radar_sub.sweep_end_ray_index['data'][sweep_idx] + 1

        # All data for a specific sweep
        sweep_data = radar_sub.fields['Edc_VEL3']['data'][start_idx:end_idx, :]

        # Finding the closest range to set ranges of data
        ranges = radar_sub.range['data']  
        closest_gate = np.argmin(np.abs(ranges - target_range))

        # Grabbing quality controlled radar data, filling in empty values with NaN
        radial_velocity = sweep_data[:, closest_gate].filled(np.nan)
        azimuths = np.linspace(start_idx, end_idx, end_idx-start_idx)
        
        try:
            # Linearly interpolating the radial velocity data where it is NaN
            valid = ~np.isnan(radial_velocity)
            radial_velocity_interp_values = np.interp(azimuths, azimuths[valid], radial_velocity[valid])
            radial_velocity_interp = np.where(valid, radial_velocity, radial_velocity_interp_values)
        except:
            radial_velocity_interp = radial_velocity
   
        # Defining fs
        fs = (end_idx - start_idx) / 360
        cutoff = 0.025

        # Padding the radial velocity data by 25 degrees to avoid weirding banding issues
        M = 25
        padded_data = np.concatenate((radial_velocity_interp[-M:], radial_velocity_interp, radial_velocity_interp[:M]))

        # Applying the lowpass filter to get the "mean" field then indexing back to the original azimuths
        radial_velocity_smoothed_padded = lowpass_filter(padded_data, cutoff=cutoff, fs=fs)
        radial_velocity_smoothed = radial_velocity_smoothed_padded[M:-M]

        # Adding to arrays and calculating residual velocity
        radial_velocity_sm[start_idx:end_idx, k] = radial_velocity_smoothed
        pertubations[start_idx:end_idx, k] = radial_velocity - radial_velocity_smoothed # (Radial velocity - "mean" = pertubations = residual velocity)

    print(f'Completed {sweep_idx+1} out of {radar.nsweeps} sweeps')

radar_sub.add_field('r_vel', {'data':pertubations})
# Optionally saving the residual velocity data before plugging it into Radx2Grid
# pyart.io.write_cfradial(f'./{rad_file[23:38]}_RVEL.nc', radar_sub)

Completed 1 out of 16 sweeps
Completed 2 out of 16 sweeps
Completed 3 out of 16 sweeps
Completed 4 out of 16 sweeps
Completed 5 out of 16 sweeps
Completed 6 out of 16 sweeps
Completed 7 out of 16 sweeps
Completed 8 out of 16 sweeps
Completed 9 out of 16 sweeps
Completed 10 out of 16 sweeps
Completed 11 out of 16 sweeps
Completed 12 out of 16 sweeps
Completed 13 out of 16 sweeps
